# Cross-validation sweep — BanglaPoliticalStance

Runs every model in `configs/` under **one** protocol: grouped stratified 5-fold CV,
augmentation inside training folds, out-of-fold predictions pooled.

**Setup:** Runtime → Change runtime type → **T4 GPU**

**6 cells, run them in order. No tokens needed.**

In [50]:
# Cell 1: GPU check
!nvidia-smi -L
import torch; print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')

GPU 0: Tesla T4 (UUID: GPU-3b30f9c8-8e7a-b62a-9d1e-8269a82ff7af)
torch 2.11.0+cu128, CUDA True


In [51]:
# Cell 2: Clone + install (repo is public)
import os, sys, subprocess, shutil
REPO = '/content/bangla-multimodal-political-stance'
os.chdir('/content')
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/kishormorol/bangla-multimodal-political-stance.git', REPO], check=True)
os.chdir(REPO)

# Install with full output so we can see errors
!pip install -e . 2>&1 | tail -10
!pip install gdown datasets 2>&1 | tail -3

# Verify
!python -c "import bmpb; print('bmpb installed OK')"
print(f'Working directory: {os.getcwd()}')

  Building editable for bmpb (pyproject.toml): finished with status 'done'
  Created wheel for bmpb: filename=bmpb-0.1.0-0.editable-py3-none-any.whl size=5576 sha256=fa2ce5140195d63a9ed07c3b9a3bdd167c0042d3b07fa41207a939f6fd0825ba
  Stored in directory: /tmp/pip-ephem-wheel-cache-9j7pui4z/wheels/40/16/33/374b9ad36acc3a4503bb6067fc765489dba7178ff3d791568d
Successfully built bmpb
  Attempting uninstall: bmpb
    Found existing installation: bmpb 0.1.0
    Uninstalling bmpb-0.1.0:
      Successfully uninstalled bmpb-0.1.0
bmpb installed OK
Working directory: /content/bangla-multimodal-political-stance


In [52]:
# Cell 3: Build corpus.csv (try Drive first, fall back to HF dataset)
import os, subprocess, sys, time
from pathlib import Path

REPO = '/content/bangla-multimodal-political-stance'
os.chdir(REPO)

# Try downloading from Drive (with retries)
for attempt in range(1, 4):
    print(f'=== bmpb data attempt {attempt}/3 ===')
    subprocess.run([sys.executable, '-m', 'bmpb.cli', 'data'])
    csvs = list(Path(REPO, 'data', 'raw').glob('*.csv'))
    if len(csvs) >= 20:
        print(f'Got {len(csvs)} CSVs.')
        break
    print(f'Got {len(csvs)} CSVs, retrying in 20s...')
    time.sleep(20)

# Try bmpb ingest
result = subprocess.run([sys.executable, '-m', 'bmpb.cli', 'ingest'],
                        capture_output=True, text=True)
print(result.stdout)

# Fallback: build corpus.csv from HF if ingest failed
corpus_path = Path(REPO, 'data', 'processed', 'corpus.csv')
if not corpus_path.exists():
    print('Ingest failed - building corpus.csv from HF dataset...')
    from datasets import load_dataset
    import pandas as pd

    # Load dataset (image decoding only triggers on __getitem__, not here)
    ds = load_dataset('kishormorol/BanglaPoliticalStance', split='annotated')
    LABEL_NAMES = {0: 'govt_critique', 1: 'neutral', 2: 'govt_leaning'}

    # Remove image column so row access never triggers PIL decoding
    ds_text = ds.remove_columns(['image'])

    # Phase 1: build corpus rows from text-only data
    print(f'Building corpus from {len(ds_text)} items (text only)...')
    rows = []
    for idx in range(len(ds_text)):
        item = ds_text[idx]
        rows.append({
            'item_id': item['item_id'],
            'title': item['headline'],
            'text': item['headline'],
            'label': item['label'],
            'label_name': LABEL_NAMES.get(item['label'], 'unknown'),
            'article_label_name': '',
            'image_label_name': '',
            'outlet': item['outlet'],
            'outlet_key': item['outlet'].lower(),
            'date': item['date'],
            'source_url': item['source_url'],
            'image_url': '',
            'image_path': '',
            'image_kind': '',
            'has_image': False,
            'annotator_1': '',
            'annotator_2': '',
            'annotator_3': '',
            'article_label': '',
            'image_label': '',
            'text_level': 'headline',
            'source_index': item['item_id'],
        })
    print(f'  Text corpus built: {len(rows)} items')

    # Phase 2: extract images from the Arrow table (raw bytes, bypasses PIL decoder)
    # image_path must be relative to data/raw/ since load_image() resolves against RAW
    images_dir = Path(REPO, 'data', 'raw', 'processed_images')
    images_dir.mkdir(parents=True, exist_ok=True)
    n_saved = 0
    n_failed = 0

    try:
        import io
        from PIL import Image as PILImage
        arrow_table = ds._data  # underlying pyarrow Table with raw bytes
        img_col = arrow_table.column('image')
        print(f'Extracting images from Arrow table ({len(img_col)} rows)...')

        for idx in range(len(img_col)):
            item_id = rows[idx]['item_id']
            try:
                img_struct = img_col[idx].as_py()
                if img_struct and img_struct.get('bytes'):
                    raw_bytes = img_struct['bytes']
                    pil_img = PILImage.open(io.BytesIO(raw_bytes))
                    img_filename = item_id + '.jpg'
                    img_save_path = images_dir / img_filename
                    pil_img.convert('RGB').save(img_save_path, format='JPEG', quality=95)
                    # Path relative to data/raw/ (load_image resolves against RAW)
                    rows[idx]['image_path'] = 'processed_images/' + img_filename
                    rows[idx]['image_kind'] = 'photo'
                    rows[idx]['has_image'] = True
                    n_saved += 1
            except Exception as e:
                n_failed += 1
                if n_failed <= 5:
                    print(f'  Skipping image for {item_id}: {e}')
        print(f'  Images: {n_saved} saved, {n_failed} failed')
    except Exception as e:
        print(f'Could not extract images: {e}')
        print('Text models will still work; multimodal models need images.')

    corpus = pd.DataFrame(rows)
    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    corpus.to_csv(corpus_path, index=False)
    print(f'Created corpus.csv: {len(corpus)} items ({n_saved} images)')

# Verify
import pandas as pd
df = pd.read_csv(corpus_path)
print(f'Corpus ready: {len(df)} items')
print(f'Labels: {df["label_name"].value_counts().to_dict()}')
n_img = int(df["has_image"].sum())
print(f'Items with images: {n_img}')
if n_img == 0:
    print('WARNING: No images found. Multimodal models will fail.')
    print('Text models (Cell 4) will work fine.')

=== bmpb data attempt 1/3 ===
Got 0 CSVs, retrying in 20s...
=== bmpb data attempt 2/3 ===
Got 0 CSVs, retrying in 20s...
=== bmpb data attempt 3/3 ===
Got 0 CSVs, retrying in 20s...

Ingest failed - building corpus.csv from HF dataset...
Building corpus from 198 items (text only)...
  Text corpus built: 198 items
Extracting images from Arrow table (198 rows)...
  Skipping image for Image_125: cannot identify image file <_io.BytesIO object at 0x7be9e9098cc0>
  Images: 86 saved, 1 failed
Created corpus.csv: 198 items (86 images)
Corpus ready: 198 items
Labels: {'govt_critique': 103, 'neutral': 53, 'govt_leaning': 42}
Items with images: 86


In [53]:
# Cell 4: Run text model CV sweep
import os
os.chdir('/content/bangla-multimodal-political-stance')
!PYTHON=$(which python) bash scripts/run_cv.sh configs/text

Cross-validating 7 configs at 5 folds

=== banglabert ===
02:19:11 INFO     banglabert: 198 all items, 5 folds                            
         INFO     no augmented table in data/raw/; training on original rows    
                  only                                                          
         INFO     device: cuda                                                  
02:19:21 INFO     HTTP Request: HEAD                                            
                  https://huggingface.co/csebuetnlp/banglabert/resolve/main/conf
                  ig.json "HTTP/1.1 307 Temporary Redirect"                     
         WARNING  Warning: You are sending unauthenticated requests to the HF   
                  Hub. Please set a HF_TOKEN to enable higher rate limits and   
                  faster downloads.                                             
         INFO     HTTP Request: HEAD                                            
                  https://huggingface.co/api/resolv

In [56]:
# Cell 5: Run multimodal model CV sweep (one at a time with full error output)
import os, subprocess, sys
os.chdir('/content/bangla-multimodal-political-stance')

# Pull latest code in case of fixes
subprocess.run(['git', 'pull', '--ff-only'], check=False)

# Pre-download all model weights before training
print('Pre-downloading model weights...')
subprocess.run([sys.executable, '-c', """
from transformers import AutoModel, AutoProcessor
models = [
    'openai/clip-vit-base-patch32',
    'kakaobrain/align-base',
    'Salesforce/blip-itm-base-coco',
    'dandelin/vilt-b32-mlm',
    'facebook/flava-full',
]
for m in models:
    print(f'  Downloading {m}...')
    try:
        AutoProcessor.from_pretrained(m)
        AutoModel.from_pretrained(m)
        print(f'    OK')
    except Exception as e:
        print(f'    FAILED: {e}')
"""])

# Clear old multimodal experiment results so leaderboard picks up fresh ones
import shutil
from pathlib import Path
for d in Path('experiments').glob('cv-align-*'):
    shutil.rmtree(d)
for d in Path('experiments').glob('cv-blip-*'):
    shutil.rmtree(d)
for d in Path('experiments').glob('cv-clip-*'):
    shutil.rmtree(d)
for d in Path('experiments').glob('cv-countvec_vit-*'):
    shutil.rmtree(d)
for d in Path('experiments').glob('cv-flava-*'):
    shutil.rmtree(d)
for d in Path('experiments').glob('cv-vilt-*'):
    shutil.rmtree(d)

PYTHON = subprocess.check_output(['which', 'python']).decode().strip()
configs = ['clip', 'align', 'blip', 'countvec_vit', 'flava', 'vilt']
failed = []
for name in configs:
    print(f'\n{"="*60}')
    print(f'=== {name} ===')
    print(f'{"="*60}')
    config_path = f'configs/multimodal/{name}.yaml'
    result = subprocess.run(
        [PYTHON, '-m', 'bmpb.cli', 'cv', '--config', config_path, '--folds', '5'],
    )
    if result.returncode != 0:
        print(f'!!! {name} FAILED (exit code {result.returncode})')
        failed.append(name)
    else:
        print(f'--- {name} OK ---')

if failed:
    print(f'\n{len(failed)} failed: {" ".join(failed)}')
else:
    print('\nAll multimodal models completed successfully!')

Pre-downloading model weights...

=== clip ===
!!! clip FAILED (exit code 1)

=== align ===
!!! align FAILED (exit code 1)

=== blip ===
!!! blip FAILED (exit code 1)

=== countvec_vit ===
--- countvec_vit OK ---

=== flava ===
--- flava OK ---

=== vilt ===
--- vilt OK ---

3 failed: clip align blip


In [55]:
# Cell 6: Leaderboard + download
import os
os.chdir('/content/bangla-multimodal-political-stance')
!python -m bmpb.cli leaderboard
print(open('reports/tables/leaderboard.md').read())

!tar czf /content/cv-runs.tar.gz experiments reports
from google.colab import files
files.download('/content/cv-runs.tar.gz')

╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/bangla-multimodal-political-stance/src/bmpb/cli.py:150 in           │
│ leaderboard                                                                  │
│                                                                              │
│   147 │   """Rebuild reports/tables/leaderboard.md."""                       │
│   148 │   from bmpb.evaluate import leaderboard as build                     │
│   149 │                                                                      │
│ ❱ 150 │   console.print(f"[green]wrote[/] {build(runs, out)}")               │
│   151                                                                        │
│   152                                                                        │
│   153 @app.command()                                                         │
│                                                                              │
│ /content/bangla-multimodal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>